In [ ]:
''' Imports '''

import pandas as pd
import polars as pl
import numpy as np
import math

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import percentileofscore
from scipy.stats.mstats import trimmed_var

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from scripts.prep_data import load_stats_team_tendencies_offense

In [ ]:
offense_tendencies = load_stats_team_tendencies_offense()

print(offense_tendencies.head().to_string())

In [ ]:
''' Features '''


FEATURES = [
    'Plays / Game', 'Drives / Game', 'Scrambles / Game', 
    '% Plays 11 Personnel', '% Under Center', 'Shotgun % Pass', 'Under Center % Pass',
    'ADOT', 'Avg Time to Throw', '% Passes Behind LOS', '% Passes Deep', 'MaxTargetShare',
    '% Rush Outside', 'MaxRushAttemptsShare',
]

# Model Preprocessing

In [ ]:
''' Transform and Scale '''
# NOTE - use standard scaling for PCA

## Scale data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(offense_tendencies[FEATURES])
scaled_data_df = pd.DataFrame(scaled_data, columns=FEATURES)

print(scaled_data_df.shape)
print(scaled_data_df.head().to_string())

# PCA

In [ ]:
''' PCA Experiment '''

# Instantiate transformer
pca = PCA(random_state=42)

# Transform data with pa
pca_component_data = pca.fit_transform(scaled_data_df)

print('Total variance:', scaled_data_df.var().sum())
print(f'Singular values:\n', pca.singular_values_)
print(f'Explained variance:\n', pca.explained_variance_.round(5))
print(f'Ratio:\n', pca.explained_variance_ratio_.round(3))
print(pca.feature_names_in_)

# Create horizontal bar chart of explained variance
fig = px.line(
    x=[i + 1 for i in range(len(pca.explained_variance_ratio_))],
    y=pca.explained_variance_ratio_.cumsum(),
    title="Explained variance"
)
fig.update_layout(xaxis_title="Principal Component", yaxis_title="Cumulative Explained Variance (%)")
fig.show()

In [ ]:
''' PCA - Final '''

# Set number of PCA components to use after initial try
PCA_N_COMPONENTS = 6
COMPONENT_COLS = [f'Component {n}' for n in range(1, PCA_N_COMPONENTS + 1)]

# Instantiate transformer
pca_final = PCA(n_components=PCA_N_COMPONENTS, random_state=42)

# Transform sku profiles
pca_component_data_final = pca_final.fit_transform(scaled_data_df)

# Evaluate components
total_variance = scaled_data_df.var().sum()
expl_variance = pca_final.explained_variance_.sum()

print(f'Data set variance: {total_variance:,.3f}')
print(f'PCA explained variance: {expl_variance:,.3f} ({round((expl_variance / total_variance) * 100, 2)}%)')

pcs = pd.DataFrame(pca_final.components_, columns=FEATURES)

# Create bar charts of contribution
for n in range(2):
    pc = pcs.transpose()[n].sort_values(ascending=False)

    fig = px.bar(
        x=pc,
        y=pc.index,
        title=f"PC{n+1}: Greatest contributors"
    )
    fig.update_layout(
        xaxis_title="Correlation", 
        yaxis_title="Features", 
        yaxis={'dtick': 1, 'categoryorder':'total ascending'},
    )
    fig.show()

    comp_expl_variance = pca_final.explained_variance_[n]
    print(f'Explained variance: {comp_expl_variance:,} ({round((comp_expl_variance / total_variance) * 100, 2)}%)')
                                                                                                                                                                                                              
# Make df of PCA scores
pca_component_df = pd.DataFrame(data=pca_component_data_final, columns=[f'Component {i}' for i in range(1, PCA_N_COMPONENTS + 1)])

print(pca_component_df.shape)
print(pca_component_df.head().to_string())

# Add PCA scores to original dataframe
offense_tendencies = offense_tendencies.drop(columns=list(filter(lambda x: x.startswith("Component"), offense_tendencies.columns)))
offense_tendencies = offense_tendencies.reset_index().merge(pca_component_df, left_index=True, right_index=True, how='left').set_index(['posteam', 'season'])

print(f'PCA values')
print(offense_tendencies.head().to_string())

In [ ]:
''' Visualize PCA Components - Scatter '''

# sl = offense_tendencies.sample(frac=0.25, random_state=42)

fig = px.scatter(
    data_frame=offense_tendencies,
    x='Component 1',
    y='Component 2',
    title='PCA Components - Top 2 Components'
)
fig.show()

fig = px.scatter_3d(
    data_frame=offense_tendencies,
    x='Component 1',
    y='Component 2',
    z='Component 3',
    title='PCA Components - Top 3 Components'
)
fig.show()

In [ ]:
''' Visualize PCA - Correlation Chart '''

# COMPONENT_NAMES = ['Small Cartons', 'High Volume', 'High Inventory - Light', 'High Inventory - Heavy']
COMPONENT_NAMES = COMPONENT_COLS

# Create "correlation matrix" - correlation with each of the components to each of the features
corr_matrix = pd.DataFrame.from_records(pca_final.components_, index=COMPONENT_NAMES, columns=FEATURES).transpose()

# Visualize
fig = px.imshow(
    corr_matrix,
    color_continuous_scale=px.colors.diverging.PRGn,
    aspect="auto"
)
fig.update_xaxes(side="top")
fig.update_coloraxes(
    showscale=False,
    cmid=0,
)
fig.update_layout(
    title='Offensive Play Style Factors',
    margin=dict(r=25, b=25, t=75),
    height=700,
    width=900

)
fig.show()

In [ ]:
''' Visualize PCA - Teams '''

# Percentile each component
for n in range(1, PCA_N_COMPONENTS + 1):
    offense_tendencies[f'Component {n} Percentile'] = offense_tendencies[f'Component {n}'].rank(pct=True)

COMPONENT_PERCENTILES = [f'Component {n} Percentile' for n in range(1, PCA_N_COMPONENTS + 1)]

def visualize_team_pca(team: str, season: int):

    ## Data ##
    # Get slice from offensive tendencies
    team_sl = offense_tendencies.loc[(offense_tendencies.index.get_level_values('posteam') == team) &
                                     (offense_tendencies.index.get_level_values('season') == season), :]
    
    # PCA Component %iles
    team_component_pct_ranks = team_sl[COMPONENT_PERCENTILES].values.tolist()[0]

    # Feature values
    team_feature_vals = team_sl[FEATURES].values.tolist()[0]
    
    # Feature value percentiles
    vals_fmt = []
    pct_scores = []
    for i in range(len(FEATURES)):
        feature = FEATURES[i]
        val = team_feature_vals[i]
        pct_score = percentileofscore(offense_tendencies[feature].tolist(), val, kind='weak') / 100
        
        val_fmt = f'{val:.1%}' if feature[0] == '%' else f'{val:.2f}'
        vals_fmt.append(val_fmt)
        pct_scores.append(f'{pct_score:.1%}')

    ## Figure ##

    fig = make_subplots(
        rows=1, cols=2, 
        column_widths=[4,3],
        horizontal_spacing=0.1,
        specs=[[{"type": "polar"}, {"type": "domain"}]]
    )

    fig.add_trace(
        go.Scatterpolar(
            r=team_component_pct_ranks,
            theta=COMPONENT_NAMES,
            opacity=0.7,
            fill='toself'
        ),
        row=1, col=1
    )
    fig.update_layout(
        title_text=f"{season} {team}",
        polar=dict(radialaxis_range=(0,1)),
        margin=dict(b=50, r=50, l=75, t=75)
    )

    fig.add_trace(
        go.Table(
            columnwidth=[2,1,1],
            header={
                "values": ['Component', 'Value', 'Percentile'],
            },
            cells={
                "values": [FEATURES, vals_fmt, pct_scores]
            }
        ),
        row=1, col=2
    )

    return fig


# visualize_team_pca('DET', 2024)
# Visualize top teams from each component
for n in range(1, PCA_N_COMPONENTS + 1):
    component = f'Component {n} Percentile'

    top_teams = offense_tendencies.sort_values(by=component, ascending=False).head(10)
    
    top_team = top_teams.index[0]
    fig = visualize_team_pca(top_team[0], top_team[1])
    fig.update_layout(
        title=f"Component {n} Top Team: {top_team[0]} {top_team[1]}"
    )
    fig.show()

    print(top_teams.to_string())